cmip6_downloader/
├── catalog/                # Catálogos Pangeo (já existentes)
│   └── pangeo-cmip6_aws.csv
│   └── pangeo-cmip6_google.csv
├── downloads/              # Diretório de saída
│   └── ...                # Arquivos .zarr e logs
├── scripts/                # Scripts principais de execução
│   └── run_download.py     # Script principal com CLI
├── src/                    # Código modularizado
│   ├── config.py           # Configurações gerais
│   ├── filters.py          # Filtros e normalização de catálogos
│   ├── preprocessing.py    # Ajustes de tempo e coordenadas
│   ├── writer.py           # Escrita incremental Zarr
│   ├── downloader.py       # Classe principal CMIP6Downloader
├── environment.yml         # Ambiente Conda reproduzível
├── README.md               # Instruções e objetivos


cmip6_downloader/
├── catalog/
├── downloads/
├── scripts/
│   └── run_download.py        ◀️ Ponto de entrada
├── src/
│   ├── config.py              ◀️ Parâmetros do projeto
│   ├── filters.py             ◀️ Filtragem do catálogo
│   ├── preprocessing.py       ◀️ Preprocessamento dos dados
│   ├── writer.py              ◀️ Escrita incremental em blocos
│   └── downloader.py          ◀️ Classe principal de orquestração
├── environment.yml            ◀️ (Podemos montar um depois)
└── README.md                  ◀️ (Também podemos gerar com instruções)


In [ ]:
# src/config.py

from pathlib import Path

# Define the root directory of the project
base_dir = Path(__file__).resolve().parent.parent

# Path to the directory containing CMIP6 catalog CSV files
catalog_dir = base_dir / "catalog"

# Path to the directory where downloaded datasets and logs will be saved
download_dir = base_dir / "downloads"

# Path to the AWS-based Pangeo CMIP6 catalog
catalog_aws = catalog_dir / "pangeo-cmip6_aws.csv"

# Path to the Google Cloud-based Pangeo CMIP6 catalog
catalog_google = catalog_dir / "pangeo-cmip6_google.csv"

# List of model names (source_id) to be included in the download
source_ids = [
    "MIROC6",
    "CMCC-ESM2",
    "ACCESS-CM2",
    "BCC-CSM2-MR",
    "INM-CM5-0",
    "EC-Earth3-Veg",
]

# List of experiment types (experiment_id), e.g., historical and future scenarios
experiment_ids = ["historical", "ssp245", "ssp585"]

# Table IDs indicating the time frequency of the data
table_ids = ["day", "3hr"]

# CMIP6 variable names to be downloaded and processed
variable_ids = [
    "tas",     # Near-surface air temperature
    "tasmax",  # Daily maximum near-surface air temperature
    "huss",    # Near-surface specific humidity
    "pr",      # Precipitation
    "wa",      # Omega: vertical velocity (pressure levels)
    "va",      # Meridional wind (pressure levels)
    "zg",      # Geopotential height (pressure levels)
    "wap",     # Pressure vertical velocity (pressure levels)
    "tos",     # Sea surface temperature
]

# Default chunk size for saving datasets in blocks along the time dimension
default_chunk_size = 5760  # ~16 years of daily or 8 months of 3-hourly data

# Standard calendar to which CFTimeIndex calendars will be converted
default_calendar = "proleptic_gregorian"


In [ ]:
# src/filters.py

import pandas as pd

# Import filtering lists from the project configuration
from src.config import (
    source_ids,
    experiment_ids,
    table_ids,
    variable_ids
)


def normalize_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize selected CMIP6 fields by stripping whitespace and ensuring string type.

    Parameters:
        df (pd.DataFrame): Input CMIP6 catalog as a DataFrame.

    Returns:
        pd.DataFrame: DataFrame with normalized fields.
    """
    # Loop through the main CMIP6 identifiers and clean them
    for col in ["source_id", "experiment_id", "member_id", "table_id", "variable_id", "version"]:
        df[col] = df[col].astype(str).str.strip()  # Ensure type str and remove extra spaces
    return df


def create_unique_key(df: pd.DataFrame) -> pd.Series:
    """
    Generate a unique string identifier for each dataset entry in the catalog.

    The key is built using a combination of CMIP6 fields.

    Parameters:
        df (pd.DataFrame): DataFrame with normalized CMIP6 metadata.

    Returns:
        pd.Series: Series of unique string keys.
    """
    return (
        df["source_id"] + "_" +
        df["experiment_id"] + "_" +
        df["member_id"] + "_" +
        df["table_id"] + "_" +
        df["variable_id"] + "_" +
        df["version"]
    )


def filter_catalog(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply project-specific filters to the CMIP6 catalog.

    Filters by source_id, experiment_id, table_id, and variable_id using values
    configured in the project.

    Parameters:
        df (pd.DataFrame): Full CMIP6 catalog DataFrame.

    Returns:
        pd.DataFrame: Filtered catalog matching the defined criteria.
    """
    df_filtered = df[
        df["source_id"].isin(source_ids) &
        df["experiment_id"].isin(experiment_ids) &
        df["table_id"].isin(table_ids) &
        df["variable_id"].isin(variable_ids)
    ].copy()

    # Reset index after filtering
    df_filtered.reset_index(drop=True, inplace=True)
    return df_filtered


In [ ]:
# src/preprocessing.py

import pandas as pd
import xarray as xr
from xarray.coding.cftimeindex import CFTimeIndex


def preprocess_time(ds: xr.Dataset, calendar: str = "proleptic_gregorian") -> xr.Dataset:
    """
    Clean and standardize the time dimension of a CMIP6 dataset.

    This function sorts time values, converts CFTimeIndex to a standard calendar,
    and removes duplicate time entries.

    Parameters:
        ds (xr.Dataset): Input dataset containing a 'time' coordinate.
        calendar (str): Target calendar format (default: 'proleptic_gregorian').

    Returns:
        xr.Dataset: Dataset with cleaned and standardized time coordinate.
    """
    # Sort by time if not already sorted
    if not ds.indexes["time"].is_monotonic_increasing:
        ds = ds.sortby("time")

    # Handle CFTimeIndex: convert calendar and drop duplicate timestamps
    if isinstance(ds.indexes["time"], CFTimeIndex):
        ds = ds.convert_calendar(calendar, align_on="year")
        time_index = pd.Index(ds.indexes["time"])
        non_duplicate_mask = ~time_index.duplicated()
        if (~non_duplicate_mask).any():
            ds = ds.isel(time=non_duplicate_mask)

    else:
        # Handle regular datetime index: drop duplicates
        time_index = ds.indexes["time"]
        non_duplicate_mask = ~time_index.duplicated()
        if (~non_duplicate_mask).any():
            ds = ds.isel(time=non_duplicate_mask)

    return ds


def adjust_coordinates_flex(ds: xr.Dataset) -> xr.Dataset:
    """
    Normalize spatial coordinates (latitude and longitude) to standard ranges.

    - Longitude is shifted to the range [-180, 180] if it is in [0, 360].
    - Latitude and longitude are sorted if 1D. For 2D (curvilinear grids), sorting is skipped.

    Parameters:
        ds (xr.Dataset): Dataset with lat/lon coordinates.

    Returns:
        xr.Dataset: Dataset with adjusted spatial coordinates.
    """
    # Attempt to infer coordinate names (supports 'lat' or 'latitude', 'lon' or 'longitude')
    lon_name = None
    lat_name = None

    for coord in ds.coords:
        if coord in ("lon", "longitude"):
            lon_name = coord
        elif coord in ("lat", "latitude"):
            lat_name = coord

    # Adjust longitude values to [-180, 180]
    if lon_name is not None:
        lon = ds[lon_name]
        if lon.ndim == 1:
            # Regular grid
            lon_new = ((lon + 180) % 360) - 180
            ds = ds.assign_coords({lon_name: lon_new})
            ds = ds.sortby(lon_name)
        elif lon.ndim == 2:
            # Curvilinear grid → only adjust, do not sort
            lon_new = ((lon + 180) % 360) - 180
            ds = ds.assign_coords({lon_name: (lon.dims, lon_new)})

    # Sort latitude if it is 1D
    if lat_name is not None:
        lat = ds[lat_name]
        if lat.ndim == 1:
            ds = ds.sortby(lat_name)
        # Do nothing for 2D latitude

    return ds


def crop_to_brazil(ds: xr.Dataset) -> xr.Dataset:
    """
    Crop the dataset to the spatial domain of Brazil (or South America, extended).

    Applies only to regular grids (1D lat/lon). If coordinates are curvilinear (2D), returns original.
    Also skips cropping for sea surface temperature ('tos') datasets.

    Parameters:
        ds (xr.Dataset): Input dataset to crop.

    Returns:
        xr.Dataset: Cropped dataset or original if not applicable.
    """
    # Skip cropping for sea surface temperature datasets
    if 'tos' in ds.data_vars:
        return ds

    lat_name = None
    lon_name = None

    # Infer coordinate names dynamically
    for coord in ds.coords:
        if coord.lower() in ("lat", "latitude"):
            lat_name = coord
        elif coord.lower() in ("lon", "longitude"):
            lon_name = coord

    # Skip cropping if lat/lon are missing or curvilinear (2D)
    if (
        lat_name is None or lon_name is None or
        ds[lat_name].ndim != 1 or ds[lon_name].ndim != 1
    ):
        return ds

    # Define slicing bounds for the Brazil region
    return ds.sel(
        **{
            lat_name: slice(-70, 20),
            lon_name: slice(-120, -30)
        }
    )


In [ ]:
# src/writer.py

import warnings
from pathlib import Path

import xarray as xr
from zarr.errors import ZarrUserWarning


def save_dataset_in_blocks(
    ds: xr.Dataset,
    output_path: Path,
    chunk_size: int = 5760
) -> None:
    """
    Save an xarray.Dataset to Zarr format in time-sliced blocks.

    This method prevents memory overload by writing small slices
    along the 'time' dimension incrementally.

    Parameters:
        ds (xr.Dataset): The xarray Dataset to save.
        output_path (Path): Destination path for the .zarr output.
        chunk_size (int): Number of time steps per block (default: 5760).
    """

    # Ensure the dataset contains a 'time' dimension
    if "time" not in ds.dims:
        print(f"⚠️ Warning: dataset {output_path.name} has no 'time' dimension. Skipping.")
        return

    # Get the total length of the time dimension
    time_len = ds.sizes["time"]

    # Loop through the dataset in chunks of size `chunk_size`
    for i in range(0, time_len, chunk_size):
        # Select and load the current time slice into memory
        block = ds.isel(time=slice(i, i + chunk_size)).load()

        # Create a new dataset with only numpy arrays (no Dask references)
        # This ensures compression consistency and avoids Zarr write issues
        ds_clean = xr.Dataset(
            data_vars={
                var: (block[var].dims, block[var].values)
                for var in block.data_vars
            },
            coords={
                coord: (block[coord].dims, block[coord].values)
                for coord in block.coords
            },
            attrs=block.attrs  # Preserve global attributes
        )

        # Determine write mode: 'w' for the first chunk, 'a' (append) for the others
        mode = "w" if i == 0 else "a"

        # Suppress expected Zarr warnings during writing
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=ZarrUserWarning)

            if mode == "w":
                # First block: create new Zarr store
                ds_clean.to_zarr(output_path, mode=mode)
            else:
                # Subsequent blocks: append along the 'time' dimension
                ds_clean.to_zarr(output_path, mode=mode, append_dim="time")

        # Log block status to console
        print(f"✔️ Saved block {i}–{min(i + chunk_size, time_len)} to {output_path}")


In [ ]:
# src/downloader.py

from pathlib import Path
from typing import Optional, Dict, List

import pandas as pd
import xarray as xr
import fsspec

from src.config import default_chunk_size, default_calendar, download_dir
from src.filters import normalize_fields, create_unique_key, filter_catalog
from src.preprocessing import preprocess_time, adjust_coordinates_flex, crop_to_brazil
from src.writer import save_dataset_in_blocks


class CMIP6Downloader:
    """
    Orchestrates the download, filtering, and block-wise saving of CMIP6 datasets
    using metadata catalogs from Pangeo CMIP6.
    """

    def __init__(
        self,
        catalog_path: Path,
        output_dir: Optional[Path] = None,
        chunk_size: int = default_chunk_size,
        calendar: str = default_calendar,
    ):
        """
        Initialize the CMIP6Downloader instance.

        Parameters:
            catalog_path (Path): Path to the CMIP6 CSV catalog (e.g., AWS or Google).
            output_dir (Optional[Path]): Directory where Zarr files will be saved.
            chunk_size (int): Number of time steps per block when saving datasets.
            calendar (str): Target calendar to standardize the time dimension.
        """
        self.catalog_path = catalog_path
        self.output_dir = output_dir or download_dir
        self.chunk_size = chunk_size
        self.calendar = calendar

        self.catalog = None  # Loaded and filtered catalog
        self.log: List[Dict] = []  # Stores processing results for each dataset

    def load_and_filter_catalog(self) -> pd.DataFrame:
        """
        Load the catalog CSV, normalize fields, filter by config, and remove duplicates.

        Returns:
            pd.DataFrame: The filtered and cleaned catalog.
        """
        df = pd.read_csv(self.catalog_path)
        df = normalize_fields(df)
        df = filter_catalog(df)

        df["key"] = create_unique_key(df)
        df = df.drop_duplicates(subset="key").copy()
        self.catalog = df

        return df

    def _open_dataset(self, row: pd.Series) -> Optional[xr.Dataset]:
        """
        Open a Zarr dataset using fsspec and xarray.

        Parameters:
            row (pd.Series): Row from the catalog containing a 'zstore' URL.

        Returns:
            xr.Dataset or None: The dataset (lazy), or None if failed to open.
        """
        zarr_url = row["zstore"]
        try:
            mapper = fsspec.get_mapper(zarr_url, anon=True)
            ds = xr.open_zarr(mapper, consolidated=True, chunks={})
            return ds
        except Exception as e:
            print(f"❌ Failed to open {zarr_url} -> {e}")
            return None

    def _format_output_name(self, row: pd.Series) -> str:
        """
        Generate a standardized file name for the output Zarr store.

        Parameters:
            row (pd.Series): Catalog row with CMIP6 metadata.

        Returns:
            str: The file name to use.
        """
        return f"{row['source_id']}_{row['experiment_id']}_{row['member_id']}_{row['table_id']}_{row['variable_id']}_{row['version']}.zarr"

    def process_row(self, row: pd.Series) -> Dict:
        """
        Process a single dataset row from the catalog: open, preprocess, and save in chunks.

        Parameters:
            row (pd.Series): One row of the CMIP6 catalog.

        Returns:
            dict: A dictionary logging the result for this dataset.
        """
        output_name = self._format_output_name(row)
        output_path = self.output_dir / output_name

        # Skip if already exists
        if output_path.exists():
            print(f"⚠️ Skipping existing file: {output_name}")
            return {"file": output_name, "status": "Skipped", "message": "Already exists"}

        try:
            ds = self._open_dataset(row)

            if ds is None:
                return {"file": output_name, "status": "Error", "message": "Failed to open dataset"}

            if "time" not in ds.dims:
                return {"file": output_name, "status": "No time", "message": "Missing 'time' dimension"}

            # Apply preprocessing steps
            ds = preprocess_time(ds, calendar=self.calendar)
            ds = adjust_coordinates_flex(ds)
            ds = crop_to_brazil(ds)

            # Save dataset in time-sliced Zarr format
            save_dataset_in_blocks(ds, output_path, chunk_size=self.chunk_size)

            return {"file": output_name, "status": "Success", "message": ""}

        except Exception as e:
            return {"file": output_name, "status": "Error", "message": str(e)}

    def run_batch(self, limit: Optional[int] = None) -> pd.DataFrame:
        """
        Run the downloader on the full catalog, optionally limited to N entries.

        Parameters:
            limit (Optional[int]): Number of rows to process (for testing/debugging).

        Returns:
            pd.DataFrame: A log of processing results for each dataset.
        """
        if self.catalog is None:
            self.load_and_filter_catalog()

        df_to_process = self.catalog if limit is None else self.catalog.tail(limit)

        for _, row in df_to_process.iterrows():
            result = self.process_row(row)
            self.log.append(result)

        return pd.DataFrame(self.log)


In [ ]:
# scripts/run_download.py

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from tqdm.auto import tqdm
import pandas as pd

from src.downloader import CMIP6Downloader
from src.config import catalog_aws, download_dir


def main():
    """
    Run the CMIP6 download pipeline using parallel threads.

    - Loads and filters the catalog.
    - Processes each dataset row using a thread pool.
    - Saves the final processing log.
    """

    # Initialize downloader with path to AWS catalog
    downloader = CMIP6Downloader(catalog_path=catalog_aws)

    # Load and filter the CMIP6 catalog
    catalog_df = downloader.load_and_filter_catalog()

    # Select how many workers (threads) to use
    max_workers = 4  # ← You can adjust this according to available CPU/memory

    # Select rows to process (optional: limit to test subset)
    rows_to_process = catalog_df.tail(2).copy()  # ← Change to full catalog_df for production

    log = []  # List to store results

    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all jobs to the thread pool
        futures = {
            executor.submit(downloader.process_row, row): idx
            for idx, (_, row) in enumerate(rows_to_process.iterrows())
        }

        # Collect results as they complete
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing CMIP6 files"):
            result = future.result()
            log.append(result)

    # Convert the log to a DataFrame
    log_df = pd.DataFrame(log)

    # Save the log to a CSV file
    log_path = download_dir / "log_download_results.csv"
    log_df.to_csv(log_path, index=False)

    print(f"\n📄 Download log saved to: {log_path}")


if __name__ == "__main__":
    main()
